# NB03 — The Shale Revolution
**Global Oil Market Analysis**

In 2008 the United States produced 5 Mb/d of oil. By 2019 it produced 12.9 Mb/d — more than Saudi Arabia or Russia. That increase, driven almost entirely by hydraulic fracturing of tight rock formations, is the biggest supply shock in oil market history. This notebook traces that transformation: formation by formation, year by year, and what it did to the global balance of power.

---

## Sections
1. US total production vs OPEC and Russia
2. The shale formations — Permian, Bakken, Eagle Ford
3. US shale share of world production over time
4. The 2014 price war — OPEC's response to shale
5. COVID collapse and shale's recovery

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.3f}'.format)

DATA_DIR = './data/processed/'

production = pd.read_parquet(DATA_DIR + 'production.parquet')
tight      = pd.read_parquet(DATA_DIR + 'tight_oil.parquet')
prices     = pd.read_parquet(DATA_DIR + 'prices.parquet')
master     = pd.read_parquet(DATA_DIR + 'master_actuals.parquet')

# Trim to actuals only
production = production.loc[:'2026-02-28']
tight      = tight.loc[:'2026-02-28']

print('Data loaded.')
print(f'Production : {production.shape}')
print(f'Tight oil  : {tight.shape}')

Data loaded.
Production : (398, 21)
Tight oil  : (314, 9)


## 1. US vs OPEC vs Russia — Who Produces the Most?

In [13]:
fig = go.Figure()

series = [
    ('prod_usa',   'United States', '#1D4ED8'),
    ('prod_opec',  'OPEC',          '#DC2626'),
    ('prod_russia','Russia',        '#7C3AED'),
    ('prod_saudi', 'Saudi Arabia',  '#D97706'),
]

for col, label, color in series:
    data = master[col].dropna() if col in master.columns else production[col.replace('prod_','')].dropna() if col.replace('prod_','') in production.columns else None
    if data is None:
        continue
    fig.add_trace(go.Scatter(
        x=data.index, y=data,
        mode='lines', name=label,
        line=dict(color=color, width=2)
    ))

# Annotate the moment US overtook Saudi Arabia
fig.add_annotation(
    x='2014-01', y=15.0,
    text='US overtakes<br>Saudi Arabia',
    showarrow=True, arrowhead=2,
    font=dict(size=11), bgcolor='white'
)

fig.update_layout(
    title='Oil Production: US vs OPEC vs Russia (Mb/d)',
    xaxis_title='Date', yaxis_title='Production (Mb/d)',
    template='plotly_white', height=480, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [3]:
# Key milestones in US production
us_prod = production['prod_usa'].dropna()
print('US PRODUCTION MILESTONES:')
print(f'  2008 trough  : {us_prod.loc["2008"].min():.2f} Mb/d ({us_prod.loc["2008"].idxmin().date()})')
print(f'  2019 peak    : {us_prod.loc["2019"].max():.2f} Mb/d ({us_prod.loc["2019"].idxmax().date()})')
print(f'  2020 COVID   : {us_prod.loc["2020"].min():.2f} Mb/d ({us_prod.loc["2020"].idxmin().date()})')
print(f'  Latest       : {us_prod.iloc[-1]:.2f} Mb/d ({us_prod.index[-1].date()})')
print()
growth_2008_2019 = (us_prod.loc['2019'].max() - us_prod.loc['2008'].min()) / us_prod.loc['2008'].min() * 100
print(f'  Growth 2008-2019: +{growth_2008_2019:.0f}%')

US PRODUCTION MILESTONES:
  2008 trough  : 7.12 Mb/d (2008-09-30)
  2019 peak    : 20.48 Mb/d (2019-12-31)
  2020 COVID   : 16.26 Mb/d (2020-05-31)
  Latest       : 23.58 Mb/d (2026-02-28)

  Growth 2008-2019: +187%


## 2. The Shale Formations — Permian, Bakken, Eagle Ford

In [4]:
# Stacked area chart of shale production by formation
formations = [
    ('tight_permian',    'Permian',      '#1D4ED8'),
    ('tight_bakken',     'Bakken',       '#059669'),
    ('tight_eagle_ford', 'Eagle Ford',   '#DC2626'),
    ('tight_niobrara',   'Niobrara',     '#D97706'),
    ('tight_austin_chalk','Austin Chalk', '#7C3AED'),
    ('tight_other',      'Other',        '#94A3B8'),
]

fig = go.Figure()
for col, name, color in formations:
    data = tight[col].dropna()
    fig.add_trace(go.Scatter(
        x=data.index, y=data,
        mode='lines', name=name,
        stackgroup='one',
        line=dict(color=color, width=0.5),
        fillcolor=color.replace('#','rgba(').replace('1D4ED8','29,78,216,0.7)') if False else color
    ))

fig.update_layout(
    title='US Tight Oil (Shale) Production by Formation (Mb/d)',
    xaxis_title='Date', yaxis_title='Production (Mb/d)',
    template='plotly_white', height=470, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [5]:
# Permian dominance — share of all shale production
tight_clean = tight.dropna(subset=['tight_total', 'tight_permian'])
tight_clean = tight_clean.copy()
tight_clean['permian_share_pct'] = tight_clean['tight_permian'] / tight_clean['tight_total'] * 100

print('PERMIAN SHARE OF ALL US SHALE PRODUCTION:')
for year in [2000, 2005, 2010, 2015, 2020, 2025]:
    try:
        val = tight_clean.loc[str(year), 'permian_share_pct'].mean()
        print(f'  {year}: {val:.1f}%')
    except Exception:
        pass

PERMIAN SHARE OF ALL US SHALE PRODUCTION:
  2000: 13.9%
  2005: 18.0%
  2010: 22.8%
  2015: 23.3%
  2020: 53.2%
  2025: 63.8%


## 3. US Shale Share of World Production

In [6]:
# Shale as % of world production over time
shale_share = master['shale_share_of_world_pct'].dropna()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['US Shale Share of World Production (%)',
                                    'World Oil Price (USD/bbl)'])

fig.add_trace(go.Scatter(
    x=shale_share.index, y=shale_share,
    mode='lines', fill='tozeroy',
    line=dict(color='#1D4ED8', width=2),
    fillcolor='rgba(29,78,216,0.12)',
    name='Shale share %'
), row=1, col=1)

price_sub = master['price_world'].dropna()
fig.add_trace(go.Scatter(
    x=price_sub.index, y=price_sub,
    mode='lines',
    line=dict(color='#DC2626', width=1.5),
    name='Price'
), row=2, col=1)

fig.update_layout(
    title='US Shale Rise vs Oil Price',
    template='plotly_white', height=560, hovermode='x unified',
    showlegend=False
)
fig.update_yaxes(title_text='Share (%)', row=1, col=1)
fig.update_yaxes(title_text='USD/bbl', row=2, col=1)
fig.show()

## 4. The 2014 Price War — OPEC's Response to Shale

In [7]:
# Zoom in: 2012-2017 — the key period of the price war
price_war = master.loc['2012':'2017'].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['Production: US vs OPEC (Mb/d)',
                                    'Oil Price (USD/bbl)'])

fig.add_trace(go.Scatter(
    x=price_war.index, y=price_war['prod_usa'],
    mode='lines', name='US', line=dict(color='#1D4ED8', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=price_war.index, y=price_war['prod_opec'],
    mode='lines', name='OPEC', line=dict(color='#DC2626', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=price_war.index, y=price_war['price_world'],
    mode='lines', name='Price',
    fill='tozeroy', fillcolor='rgba(220,38,38,0.1)',
    line=dict(color='#DC2626', width=1.5)
), row=2, col=1)

# Annotate Nov 2014 — OPEC decides to defend market share not price
fig.add_vline(x='2014-11-27', line_dash='dash', line_color='orange',
              line_width=2, opacity=0.8)
fig.add_annotation(
    x='2014-11-27', y=35, row=2, col=1,
    text='Nov 2014:<br>OPEC defends<br>market share',
    showarrow=False, font=dict(size=10, color='darkorange'),
    xanchor='left', bgcolor='white'
)

fig.update_layout(
    title='2014 Oil Price War — OPEC vs US Shale',
    template='plotly_white', height=560, hovermode='x unified',
    legend=dict(orientation='h', y=-0.12)
)
fig.show()

In [ ]:
# Did OPEC actually increase production to kill shale?
print('OPEC PRODUCTION DURING THE PRICE WAR:')
for month in ['2014-06', '2014-11', '2015-06', '2015-12', '2016-06', '2016-12']:
    try:
        val   = master.loc[month, 'prod_opec'].iloc[0]
        price = master.loc[month, 'price_world'].iloc[0]
        print(f'  {month}: OPEC={val:.2f} Mb/d | Price=${price:.2f}/bbl')
    except Exception as e:
        print(f'  {month}: missing ({e})')

OPEC PRODUCTION DURING THE PRICE WAR:
  2014-06: OPEC=32.65 Mb/d | Price=$108.37/bbl
  2014-11: OPEC=33.20 Mb/d | Price=$76.99/bbl
  2015-06: OPEC=34.54 Mb/d | Price=$61.31/bbl
  2015-12: OPEC=34.77 Mb/d | Price=$36.57/bbl
  2016-06: OPEC=35.26 Mb/d | Price=$47.69/bbl
  2016-12: OPEC=36.01 Mb/d | Price=$52.62/bbl


## 5. COVID Collapse and Shale Recovery

In [9]:
# 2019-2026: collapse and recovery
covid_era = master.loc['2019':'2026'].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['US Shale Production by Key Formation (Mb/d)',
                                    'Oil Price (USD/bbl)'])

for col, label, color in [
    ('prod_tight_permian', 'Permian',    '#1D4ED8'),
    ('prod_tight_bakken',  'Bakken',     '#059669'),
    ('prod_tight_eagle_ford','Eagle Ford','#DC2626'),
]:
    data = covid_era[col].dropna()
    fig.add_trace(go.Scatter(
        x=data.index, y=data,
        mode='lines', name=label,
        line=dict(color=color, width=2)
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=covid_era.index, y=covid_era['price_world'],
    mode='lines', fill='tozeroy', fillcolor='rgba(220,38,38,0.08)',
    line=dict(color='#DC2626', width=1.5), name='Price',
    showlegend=False
), row=2, col=1)

fig.add_vrect(x0='2020-03', x1='2020-06',
              fillcolor='red', opacity=0.07,
              annotation_text='COVID crash', annotation_position='top left')

fig.update_layout(
    title='Shale Production — COVID Collapse and Recovery (2019-2026)',
    template='plotly_white', height=560, hovermode='x unified',
    legend=dict(orientation='h', y=-0.12)
)
fig.show()

In [10]:
# Summary findings
tight_clean = tight.dropna(subset=['tight_total'])
print('=== NB03 KEY FINDINGS ===')
print()
print('US shale production:')
print(f'  2000: {tight_clean.loc["2000", "tight_total"].mean():.2f} Mb/d')
print(f'  2010: {tight_clean.loc["2010", "tight_total"].mean():.2f} Mb/d')
print(f'  2015: {tight_clean.loc["2015", "tight_total"].mean():.2f} Mb/d')
print(f'  2019 peak: {tight_clean.loc["2019", "tight_total"].max():.2f} Mb/d')
print(f'  2020 COVID low: {tight_clean.loc["2020", "tight_total"].min():.2f} Mb/d')
print(f'  Latest: {tight_clean["tight_total"].iloc[-1]:.2f} Mb/d ({tight_clean.index[-1].date()})')

print()
print('Permian formation:')
perm_2010 = tight_clean.loc['2010','tight_permian'].mean()
perm_late = tight_clean['tight_permian'].iloc[-12:].mean()
print(f'  2010 avg: {perm_2010:.2f} Mb/d')
print(f'  Latest avg (12-mo): {perm_late:.2f} Mb/d')
print(f'  Growth: +{(perm_late/perm_2010 - 1)*100:.0f}%')

print()
print('NB03 complete. Proceed to NB04 — Supply, Demand, and the Market Mechanism.')

=== NB03 KEY FINDINGS ===

US shale production:
  2000: 0.25 Mb/d
  2010: 0.68 Mb/d
  2015: 4.58 Mb/d
  2019 peak: 8.21 Mb/d
  2020 COVID low: 6.09 Mb/d
  Latest: 9.23 Mb/d (2026-02-28)

Permian formation:
  2010 avg: 0.15 Mb/d
  Latest avg (12-mo): 5.95 Mb/d
  Growth: +3777%

NB03 complete. Proceed to NB04 — Supply, Demand, and the Market Mechanism.
